MAKEMORE_II: CHARACTER LANGUAGE MODEL

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline 

In [2]:
words = open('names.txt','r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
block_size = 3
X, Y = [],[]
for w in words[:5]:
    print(w)
    context = [0]*block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)  

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

Lookup Table C: (Input layer)

In [11]:
C = torch.randn((27, 2))

In [16]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

First Hidden Layer: Tanh

In [ ]:
W1 = torch.randn((6,100)) #6 inputs and 100 neurons
b1 = torch.randn(100) #each neuron has a bias

In [20]:
torch.cat([emb[:,0,:], emb[:,1,:],emb[:,2,:]],1).shape #cat concatenates the first 3 embeddings

torch.Size([32, 6])

In [ ]:
torch.cat(torch.unbind(emb, 1),1).shape #unbind helps us do the same thing as previous line, but without having to call emb slicing again and again.

torch.Size([32, 6])

In [31]:
emb.view(32,6).shape #View helps us view the same tensor in different dimensions. Therefore this is the same as the previous line, but more efficient as cat creates another memory space, view doesnt

torch.Size([32, 6])

In [ ]:
#Therefore
h = torch.tanh(emb.view(-1,6) @ W1 + b1) #tanh layer

In [34]:
h

tensor([[ 0.9993, -0.9153, -0.8126,  ..., -0.9694, -0.7648, -0.8327],
        [ 0.9974, -0.9887, -0.4966,  ..., -0.9206, -0.4814,  0.6394],
        [ 0.7037, -0.4373, -0.3844,  ..., -0.0184, -0.9977, -0.6334],
        ...,
        [ 1.0000,  0.9412,  0.4799,  ..., -0.9990, -0.9997,  0.8057],
        [ 0.1588,  0.1706, -0.0620,  ..., -0.9867,  0.7025,  0.5778],
        [ 0.9995, -1.0000, -0.9936,  ..., -0.3407,  0.7190, -0.9982]])

In [35]:
h.shape

torch.Size([32, 100])

Final Layer: Softmax

In [36]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)

In [37]:
logits = h @ W2 + b2

In [38]:
logits.shape

torch.Size([32, 27])

In [39]:
counts = logits.exp()

In [40]:
prob = counts/counts.sum(1,keepdims=True)

In [43]:
prob.shape

torch.Size([32, 27])

In [44]:
prob[0].sum()

tensor(1.)

In [ ]:
loss = -prob[torch.arange(32), Y].log().mean()
loss #negative log likelihood

tensor(16.2474)

Neural Network Summary:

Getting Input Data and Labels:

In [49]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

Getting the parameters: C,W1,b1,W2,b2

In [50]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100,generator=g)
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

Total number of params:

In [51]:
sum(p.nelement() for p in parameters)

3481

Hidden and output Layer: Tanh, Softmax

In [53]:
#Hidden:
emb = C[X] #(32,3,2)
h = torch.tanh(emb.view(-1,6) @ W1 + b1) #(32,100)

#Output:
logits = h @W2 +b2 #(32,27)
counts = logits.exp()
prob = counts/counts.sum(1,keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)